# 03 — Signal research

**What this does:** reruns the alignment harness and evaluates momentum, value, quality, capital deployment, risk, short-interest sentiment, and fundamental momentum using the shared research machinery.

**Run first:** notebook 01 must have produced a valid `data/panel_russell3000.parquet`. Monthly notebook 02 is not required unless you want the latest provisional months included.

**Produces:** printed research tables and charts in this notebook. It does not change the panel or production portfolio. The permanent written decision record is `docs/signal_log.md`.

**Time:** approximately 3–8 minutes. Some quarterly rolling calculations can be quiet for a minute.

## Step 1 — Load the research environment

This locates the project, loads the shared evaluator, and displays the settings that govern the sample and standardization.

In [ ]:
%matplotlib inline
from pathlib import Path
import sys

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'config.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not find the project folder containing config.py.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from IPython.display import display
import importlib
import config
config = importlib.reload(config)  # Refresh settings in a reused notebook kernel.
from src import signal_evaluation as evaluate
from src.notebook_support import confirm_evaluation, confirm_harness, settings_table

display(settings_table())
print('Signal research environment loaded. All calculations below call src/signal_evaluation.py.')

## Step 2 — Test alignment, then evaluate 12-1 momentum

The random tests should find nothing, while perfect foresight should produce rank IC near one. Any harness failure is a stop condition: signal results cannot be trusted if forward returns are misaligned.

In [ ]:
momentum_research = evaluate.run_signal_research()
display(confirm_harness(momentum_research['harness']))
display(confirm_evaluation('12-1 momentum', momentum_research['momentum']))

## Step 3 — Evaluate value

Value combines cash-flow yield, earnings yield, and EBITDA to enterprise value. Financials and Real Estate omit EBITDA/EV but retain the other value measures.

In [ ]:
value_research = evaluate.run_value_research()
display(confirm_evaluation('Value block', value_research['evaluations']['value_block']))

## Step 4 — Evaluate quality

Quality reports all six tested components, the original six-signal block, and the final three-input block. Asset-turnover and EPS stability remain visible even though they were rejected.

In [ ]:
quality_research = evaluate.run_quality_research()
display(confirm_evaluation('Revised quality block', quality_research['evaluations']['quality_block']))

## Step 5 — Evaluate capital deployment

This first audits split adjustment, then evaluates share-count reduction over three, twelve, and thirty-six months and their combined block.

In [ ]:
capital_research = evaluate.run_capital_deployment_research()
display(confirm_evaluation('Capital-deployment block', capital_research['evaluations']['capital_deployment_block']))

## Step 6 — Reproduce the rejected risk block

Risk is deliberately retained as a failed experiment. Low volatility, low beta, and low idiosyncratic volatility produced inverted or weak spreads, so this block is **not** part of the model.

In [ ]:
risk_research = evaluate.run_risk_research()
display(confirm_evaluation('Risk block — rejected', risk_research['evaluations']['risk_block']))

## Step 7 — Evaluate short-interest sentiment

The sentiment score ranks lower split-adjusted short interest as better. Missing reports remain missing rather than being interpreted as zero.

In [ ]:
sentiment_research = evaluate.run_sentiment_research()
display(confirm_evaluation('Short-interest sentiment block', sentiment_research['evaluation']))

## Step 8 — Reproduce the rejected fundamental-momentum block

This evaluates cash-flow and revenue trends and the predeclared four-, eight-, and twelve-quarter robustness check. The effect decayed as the window lengthened, so the block is **not** part of the model.

In [ ]:
fundamental_momentum_research = evaluate.run_fundamental_momentum_research()
display(confirm_evaluation(
    'Fundamental-momentum block — rejected',
    fundamental_momentum_research['evaluations']['fundamental_momentum_block'],
))

## Step 9 — Contributor extension: evaluate a newly registered block

After copying the signal template and adding its `SignalBlockSpec` to `src/signals/registry.py`, put that registry name in the cell below. The generic evaluator runs the standard component and block tests without a new hand-written runner. Leave it as `None` during the normal reproduction run. Record every result in `docs/signal_log.md`.

In [ ]:
REGISTERED_BLOCK_TO_EVALUATE = None  # Example: 'new_block_name'

if REGISTERED_BLOCK_TO_EVALUATE is None:
    print('No contributor block requested. Set REGISTERED_BLOCK_TO_EVALUATE after registering one.')
else:
    registered_research = evaluate.run_registered_block_research(
        REGISTERED_BLOCK_TO_EVALUATE
    )
    registered_spec = registered_research['spec']
    display(confirm_evaluation(
        registered_spec.label,
        registered_research['evaluations'][registered_spec.block_attribute],
    ))

## What was produced and what comes next

This notebook reproduced the forward-alignment harness and every tested signal block. It wrote no production files. Read `docs/signal_log.md` for the permanent decision record and the reason each signal was kept, combined, or rejected.

To see how the five accepted blocks combine and behave in a portfolio, continue to **04_backtest.ipynb**.

## If something fails

- **Missing panel:** run notebook 01 and confirm its panel step passed.
- **Harness failure:** stop; do not interpret any signal output. Save the harness table and investigate the failure.
- **Memory or dead kernel:** close other notebooks, restart Jupyter, and run this notebook alone.
- **Undefined result variable:** restart the kernel and run from Step 1.
- **Unexpectedly different result:** confirm `config.py` settings and the panel date range before changing any signal. See `START_HERE.ipynb` for detailed troubleshooting.